# Tokenización

¡Bienvenido a tu primer laboratorio práctico de Procesamiento de Lenguaje Natural (NLP)! 
A diferencia de las imágenes, que son naturalmente arreglos numéricos, el texto es una secuencia de símbolos que las máquinas no entienden intrínsecamente. Antes de poder realizar tareas como el análisis de sentimientos o la traducción, primero debes convertir tu texto a un formato que un modelo pueda procesar.


La **tokenización** es un primer paso importante en cualquier flujo de trabajo de NLP, ya que convierte el texto bruto en unidades significativas llamadas **tokens**.
Estos tokens son las piezas de construcción utilizadas por los modelos, como BERT, para generar incrustaciones de palabras (*word embeddings*): representaciones vectoriales densas que capturan el significado semántico.


Este laboratorio ofrece una visión práctica de este proceso fundamental. 
Explorarás la tokenización comparando un enfoque manual desde cero con el uso de una herramienta moderna y preentrenada.

Específicamente, aprenderás a:
* Construir un tokenizador simple desde cero para entender la mecánica principal, creando un vocabulario donde cada palabra única se mapea a un ID numérico.
* Utilizar un potente tokenizador BERT preentrenado de la popular librería Hugging Face para ver cómo los profesionales gestionan esta tarea de manera eficiente.
* Comprender por qué es crítico emparejar los tokenizadores con los modelos y usar `AutoTokenizer` como una mejor práctica para asegurar la compatibilidad.
* Observar cómo esta herramienta avanzada maneja automáticamente desafíos como las palabras fuera del vocabulario (OOV) dividiéndolas en **tokens de subpalabras** (*subword tokens*).

## Imports

In [ ]:
import torch
from transformers import BertTokenizerFast, AutoTokenizer

import helper_utils

## Tokenización manual: Construyendo un vocabulario

* Define una lista de oraciones de ejemplo (`sentences`).


In [ ]:
sentences = [
    'I love my dog',
    'I love my cat'
]

* Implementa la función `tokenize` que convierte el texto de entrada (`text`) a minúsculas y lo divide en palabras individuales (tokens) basándose en los espacios en blanco.

In [ ]:
def tokenize(text):
    """
    Tokeniza el texto proporcionado convirtiéndolo a minúsculas 
    y dividiéndolo por espacios en blanco.

    Args:
        text: La cadena de entrada que será tokenizada.

    Returns:
        tokens: Una lista de tokens (cadenas de texto) en minúsculas derivados 
                del texto de entrada.
    """
    # Convertir el texto a minúsculas y dividir por espacios en blanco
    tokens = text.lower().split()

    return tokens

* Implementa la función `build_vocab` que toma una lista de oraciones, las tokeniza y crea un vocabulario (`vocab`).
    * Cada palabra única encontrada se añade al vocabulario y se le asigna un ID numérico único.

In [ ]:
def build_vocab(sentences):
    """
    Construye un mapeo de vocabulario a partir de una lista de oraciones, asignando 
    un ID entero único a cada token único.

    Args:
        sentences: Una lista de cadenas de texto que representan el corpus a 
                   ser procesado.

    Returns:
        vocab: Un diccionario que mapea tokens de texto únicos a índices 
               enteros únicos.
    """
    vocab = {}

    # Iterar sobre la lista de oraciones de entrada
    for sentence in sentences:
        # Convertir la oración en una lista de tokens usando el tokenizador
        tokens = tokenize(sentence)

        # Procesar cada token individual encontrado en la oración
        for token in tokens:
            # Verificar si el token falta actualmente en el diccionario de vocabulario
            if token not in vocab:
                # Asignar un nuevo índice entero único al token, comenzando desde 1
                vocab[token] = len(vocab) + 1
    
    return vocab

In [ ]:
# Create the vocabulary index
vocab = build_vocab(sentences)

print("Vocabulary Index:", vocab, "\n")  

## Uso de un tokenizador BERT preentrenado

* Inicializa el `BertTokenizerFast` cargando el modelo preentrenado [bert-base-uncased](https://huggingface.co/google-bert/bert-base-uncased) directamente desde Hugging Face.



**Nota**: En este entorno de notebook, el modelo se ha guardado y se está cargando localmente:

```python
local_tokenizer_path = "./bert_tokenizer_local"
tokenizer = BertTokenizerFast.from_pretrained(local_tokenizer_path)
```

Si ejecutaras este notebook en otro lugar, lo inicializarías como:
     
```python
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
```

* Utiliza el `tokenizer` inicializado para procesar las oraciones (`sentences`), creando las entradas codificadas (`encoded_inputs`).
    * `padding=True` asegura que todas las secuencias de salida tengan la misma longitud.
    * `truncation=True` corta las secuencias que son más largas que la longitud máxima de entrada del modelo.
    * `return_tensors='pt'` especifica que la salida deben ser tensores de PyTorch.



* Convierte los `input_ids` (representaciones numéricas) de `encoded_inputs` de nuevo a sus representaciones de tokens de cadena para facilitar la inspección.
    * Estos pueden incluir tokens especiales como `[CLS]` y `[SEP]`.
* Recupera todo el vocabulario (mapeo de palabra a ID) utilizado por el tokenizador BERT mediante `tokenizer.get_vocab()`.
* Imprime los `input_ids` (IDs de tokens) generados por el tokenizador para las oraciones.

In [ ]:
sentences = [
    'I love my dog',
    'I love my cat'
]

# Definir el directorio local donde está guardado el tokenizador
local_tokenizer_path = "./bert_tokenizer_local"

# Inicializar el tokenizador desde el directorio local
tokenizer = BertTokenizerFast.from_pretrained(local_tokenizer_path)

# Tokenizar las oraciones y codificarlas
encoded_inputs = tokenizer(sentences, padding=True, 
                           truncation=True, return_tensors='pt')

# Ver los tokens de cada entrada (útil para entender la salida)
tokens = [tokenizer.convert_ids_to_tokens(ids)
          for ids in encoded_inputs["input_ids"]]

# Obtener el vocabulario del modelo (mapeo de tokens a IDs)
word_index = tokenizer.get_vocab() # Para BertTokenizerFast, get_vocab() devuelve el vocabulario

# Imprimir los tokens legibles para humanos de cada oración
print("Tokens:", tokens)

print("\nToken IDs:", encoded_inputs['input_ids'])

# Imprimir el mapeo de tokens únicos de tus oraciones a sus IDs únicos 
helper_utils.print_unique_token_id_mappings(tokens, encoded_inputs['input_ids'])

**Observación sobre la compatibilidad del modelo**
Vale la pena enfatizar que en NLP, los tokenizadores no son herramientas de "talla única". Cada tokenizador está diseñado específicamente para funcionar con un modelo en particular.
El tokenizador `bert-base-uncased`, por ejemplo, está diseñado para dar formato al texto de la manera exacta en que el modelo BERT fue entrenado para entenderlo. 
Esto incluye su vocabulario específico, reglas para dividir palabras y el uso de tokens especiales como `[CLS]` y `[SEP]`.



*El uso del tokenizador que coincide con su modelo asegura que el formato de entrada sea exactamente lo que el modelo espera.* El desajuste de un modelo y un tokenizador puede conducir a un rendimiento deficiente o errores.

## Uso de `AutoTokenizer`

Si bien el uso de una clase específica como `BertTokenizerFast` funciona perfectamente, la biblioteca `transformers` de Hugging Face ofrece una solución conveniente y robusta: `AutoTokenizer`.

La clase `AutoTokenizer` es un contenedor inteligente que detecta y carga automáticamente la clase de tokenizador correcta para cualquier punto de control (checkpoint) de modelo dado. 
En lugar de que necesites recordar si un modelo requiere `BertTokenizerFast`, `GPT2Tokenizer` u otra clase específica, `AutoTokenizer.from_pretrained()` lo gestiona por ti.

Esto simplifica tu código y, lo que es más importante, evita posibles desajustes entre tu modelo y su tokenizador. 

* Inicializa el `AutoTokenizer` cargando el mismo modelo preentrenado `bert-base-uncased`.
* Utiliza el `AutoTokenizer` para procesar las oraciones (`sentences`), creando `encoded_inputs`.
    * Se utilizan los mismos parámetros (`padding`, `truncation`, `return_tensors`) para garantizar un formato de salida consistente.
* Convierte los `input_ids` de `encoded_inputs` de nuevo a sus representaciones de tokens de cadena para su inspección.
* Imprime los `input_ids` generados por el `AutoTokenizer` para las oraciones.

In [ ]:
# Definir el directorio local donde está guardado el tokenizador
local_tokenizer_path = "./bert_tokenizer_local"

# Inicializar el tokenizador usando la clase AutoTokenizer
# Esto carga automáticamente el tokenizador correcto (BertTokenizerFast en este caso)
tokenizer = AutoTokenizer.from_pretrained(local_tokenizer_path)

In [ ]:
sentences = [
    'I love my dog',
    'I love my cat'
]

# Tokenizar las oraciones y codificarlas
encoded_inputs = tokenizer(sentences, padding=True, 
                           truncation=True, return_tensors='pt')

# Ver los tokens de cada entrada (útil para entender la salida)
tokens = [tokenizer.convert_ids_to_tokens(ids)
          for ids in encoded_inputs["input_ids"]]

# Obtener el vocabulario del modelo (mapeo de tokens a IDs)
word_index = tokenizer.get_vocab() 

# Imprimir los tokens legibles para humanos de cada oración
print("Tokens:", tokens)

print("\nToken IDs:", encoded_inputs['input_ids'])

# Imprimir el mapeo de tokens únicos de tus oraciones a sus IDs únicos 
helper_utils.print_unique_token_id_mappings(tokens, encoded_inputs['input_ids'])

## (Opcional) Pruébalo con tus propias oraciones

Has visto cómo el tokenizador BERT preentrenado procesó las oraciones de ejemplo. ¡Ahora es tu turno de experimentar! Usa la celda de código a continuación para ingresar tus propias oraciones y observa cómo se tokenizan.

Prueba oraciones de diferentes longitudes. Por ejemplo:

```python
sentences = [
    'I love my red dog',
    'I love my cat'
]
```

In [ ]:
### Add your sentence(s) here               
sentences = [
    "",
    # "",
    # "",
]

La siguiente celda de código está lista para tomar estas oraciones y procesarlas utilizando el tokenizador. Luego, imprimirá cómo tus oraciones han sido convertidas en 'Tokens' y sus correspondientes 'Token IDs'.

**Antes de ejecutarla, ten en cuenta lo siguiente:** Si has incluido palabras que son particularmente únicas o específicas (como nombres de personas locales, lugares específicos o sustantivos menos comunes), presta mucha atención a cómo aparecen estas palabras en la lista de 'Tokens' después de ejecutar la celda. Es posible que notes que se manejan de una manera distinta.

Se proporcionará una explicación para este comportamiento, especialmente en relación con este tipo de palabras, en la sección debajo de la salida, bajo el encabezado **Palabras "fuera del vocabulario" (Out-of-Vocabulary)**.

In [ ]:
# Tokenizar las oraciones y codificarlas
encoded_inputs = tokenizer(sentences, padding=True, 
                           truncation=True, return_tensors='pt')

# Ver los tokens de cada entrada (útil para entender la salida)
tokens = [tokenizer.convert_ids_to_tokens(ids)
          for ids in encoded_inputs["input_ids"]]

# Obtener el vocabulario del modelo (mapeo de tokens a IDs)
word_index = tokenizer.get_vocab()

# Imprimir los tokens legibles para humanos de cada oración
print("Tokens:", tokens)

print("\nToken IDs:", encoded_inputs['input_ids'])

# Imprimir el mapeo de tokens únicos de tus oraciones a sus IDs únicos 
helper_utils.print_unique_token_id_mappings(tokens, encoded_inputs['input_ids'])

### Palabras "fuera del vocabulario" (OOV)

Es posible que hayas visto algunas palabras en tus oraciones (especialmente nombres únicos o términos locales) dividirse en fragmentos más pequeños al ser tokenizadas. Esto es lo esperado.

* **¿Qué son las palabras OOV?** Son palabras que no están en el diccionario integrado del tokenizador (por ejemplo, muchos nombres propios).
* **¿Cómo se manejan?** El tokenizador divide las palabras OOV en partes de subpalabras más pequeñas y conocidas.
* **¿Qué significa "##"?** Una subpalabra que comienza con "##" (como ##bs) se adjunta a la pieza anterior para formar la palabra original. No es una palabra nueva por sí misma.

**Ejemplo**:
Si un nombre como `"Mubsi"` es OOV, podría convertirse en ['mu', '##bs', '##i']. Esto significa que "mu" + "bs" + "i" se combinan para representar "Mubsi".

**¿Por qué sucede esto?** Esta "tokenización de subpalabras" permite que el tokenizador maneje cualquier palabra, incluso si es rara o nueva, asegurando que ninguna palabra sea realmente "desconocida".

Para ver esto en acción, utiliza el `tokenizer` en las `oov_words` y revisa los tokens de salida.

In [ ]:
# Una lista de palabras que probablemente están "Fuera del Vocabulario" (OOV)
oov_words = ["Tokenization", "HuggingFace", "unintelligible"]

print("--- Ejemplo de Tokenización de Subpalabras ---")

# Iterar a través de las palabras y mostrar cómo se tokenizan
for word in oov_words:
    # El método .tokenize() es una forma directa de ver el desglose de subpalabras
    subwords = tokenizer.tokenize(word)
    
    # Imprimir los resultados
    print(f"Palabra original: '{word}'")
    print(f"Tokens de subpalabras: {subwords}\n")

# Conclusión

¡Felicidades por completar el laboratorio! Has transformado con éxito texto bruto en tensores numéricos estructurados que un modelo de aprendizaje profundo puede entender.

Comenzaste construyendo un vocabulario manualmente, tokenizando oraciones y asignando un ID único a cada palabra. Este ejercicio fundamental resalta el desafío principal: cada palabra única necesita una representación numérica, y tu vocabulario puede volverse masivo y difícil de manejar rápidamente.

Luego, viste el enfoque moderno: el uso de un tokenizador preentrenado. Con solo unas pocas líneas de código, este maneja todo el flujo de trabajo de preprocesamiento, desde la división de palabras y la adición de tokens especiales como `[CLS]` y `[SEP]`, hasta el relleno (padding) y el truncamiento. También viste cómo la **tokenización de subpalabras** resuelve elegantemente el problema de las palabras fuera del vocabulario, asegurando que ninguna palabra sea realmente "desconocida" para el modelo.

La conclusión clave es que el tokenizador y su modelo correspondiente están estrechamente vinculados. El tokenizador BERT da formato al texto de la manera exacta en que el modelo BERT fue entrenado para entenderlo, lo cual es esencial para lograr un rendimiento de vanguardia. El uso de herramientas como `AutoTokenizer` simplifica este proceso, garantizando que siempre utilices el tokenizador correcto para el modelo elegido. Ahora que puedes convertir de manera confiable cualquier texto en tensores listos para el modelo, estás preparado para pasar a la siguiente etapa: usar estos tensores para construir y entrenar potentes modelos de NLP.